# Hướng Dẫn Fine-Tuning LLM Bằng Kaggle GPU Miễn Phí (Unsloth + QLoRA)

Notebook này được thiết kế để bạn chạy trực tiếp trên **Kaggle** (miễn phí 30 giờ GPU T4/P100 mỗi tuần).

### Các bước chuẩn bị trên Kaggle:
1. Truy cập `https://www.kaggle.com` và tạo tài khoản (xác minh số điện thoại để bật GPU miễn phí).
2. Bấm **+ Create** -> **New Notebook**.
3. Ở cột bên phải (**Notebook options**):
   - **Accelerator**: Chọn **GPU T4 x 2** (hoặc GPU T4).
   - **Internet**: Bật **Internet On**.
4. Tải file notebook này lên và chạy từng ô bên dưới.

In [ ]:
# 1. Cài đặt Unsloth và các thư viện cần thiết
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes

In [ ]:
# 2. Nạp mô hình cơ sở ở định dạng QLoRA 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Tự động chọn Float16/Bfloat16
load_in_4bit = True # Bật 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Gắn LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Chuẩn bị dữ liệu mẫu tiếng Việt (Bạn có thể tải dataset.jsonl của riêng bạn lên Kaggle)
from datasets import Dataset

sample_data = [
    {
        "instruction": "Bạn là ai và làm việc cho ai?",
        "input": "",
        "output": "Tôi là trợ lý AI cá nhân độc lập của bạn, hoàn toàn riêng tư và trung thành tuyệt đối với bạn."
    },
    {
        "instruction": "Quy tắc trả lời ngôn ngữ của bạn là gì?",
        "input": "",
        "output": "Tôi luôn luôn trả lời bằng tiếng Việt tự nhiên, chính xác, không dùng tiếng Anh trừ khi bạn yêu cầu."
    },
    {
        "instruction": "Hãy viết một đoạn mã Python đọc file json.",
        "input": "",
        "output": "```python\nimport json\n\nwith open('data.json', 'r', encoding='utf-8') as f:\n    data = json.load(f)\nprint(data)\n```"
    }
]

dataset = Dataset.from_list(sample_data)

alpaca_prompt = """Dưới đây là một hướng dẫn mô tả nhiệm vụ. Hãy viết phản hồi hoàn thành yêu cầu một cách chính xác.

### Hướng dẫn:
{}

### Đầu vào:
{}

### Phản hồi:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 5. Huấn luyện (Training)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
# 6. Xuất trực tiếp ra file GGUF để tải về nạp vào Bionic
# File sẽ được lưu trong thư mục làm việc của Kaggle và bạn có thể bấm Download về máy tính
model.save_pretrained_gguf("model_tieng_viet", tokenizer, quantization_method = "q4_k_m")
print("Hoàn tất xuất file GGUF!")